In [ ]:
def unpack(xys):
    return [xy[0] for xy in xys], [xy[1] for xy in xys]

In [ ]:
# reported in Corley et al, 1971
# year - trunk mass

xys = [[1.5, 4.7],
        [2.5, 13.8],
        [4.5, 43.6],
        [6.5, 78.9],
        [8.5, 103.7],
        [10.5, 145.3],
        [14.5, 232.6],
        [17.5, 290.7],
        [27.5, 300.5],]

xs = [xy[0] for xy in xys]
ys = [xy[1] for xy in xys]

import matplotlib.pyplot as plt
%matplotlib inline

from scipy.optimize import curve_fit
import numpy as np

def g(x,a,b,c):
    y = a*np.exp(-b*np.exp(-c*x))
    return y

params, _ = curve_fit(g, xs, ys)
ygs = [g(x,*params) for x in xs]

fig, ax = plt.subplots()

ax.scatter(xs,ys, color='black', label='observed')

xms = np.arange(0,30,1)
ygs = [g(x,*params) for x in xms]

ax.plot(xms,ygs, color='black', label='model-fit')

ax.legend()

dydxs = []

dx = 0.1

for x in xms:
    num = g(x+dx, *params) - g(x, *params)
    denum = dx
    dydx = num/denum
    dydxs += [dydx]

ls = [[x,round(y,1)] for x,y in zip(xms[::3], dydxs[::3])]

In [ ]:
# reported in Corley et al, 1971
# year - root mass

xys = [[1.5, 8.8],
        [2.5, 15.9],
        [4.5, 36.4],
        [6.5, 40.6],
        [8.5, 44.0],
        [10.5, 49.0],
        [14.5, 68.9],
        [17.5, 61.6],]
      #[27.5, 130.8]]

outliers = []

c = 1000/144

other_africa = [[17, c*16.4], [10, c*31.5]]
other_indo = [[8, c*14.1], [8, c*9.7]]

import matplotlib.pyplot as plt
%matplotlib inline

from scipy.optimize import curve_fit
import numpy as np

def g(x,a,b,c):
    y = a*np.exp(-b*np.exp(-c*x))
    return y

def f(x, a):
    return a*x

opt = 'g'

if opt == 'g':
    func = g
    p0 = [93.92359701,  2.08304381,  0.11793905]
else:
    func = f
    p0 = [1]
    
params, _ = curve_fit(func, *unpack(xys), p0=p0)
yfs = [func(x,*params) for x in xs]

fig, ax = plt.subplots()

ax.scatter(*unpack(xys), color='black', label='observed')

ax.scatter(*unpack(outliers), color='black', marker= 'x',
           label='observed - outlier')

ax.scatter(*unpack(other_africa), color='black', marker= '^',
           label='other - africa')

ax.scatter(*unpack(other_indo), color='black', marker= 'v',
           label='other - indo')

xms = np.arange(0,30,1)
yfs = [func(x,*params) for x in xms]

ax.plot(xms,yfs, color='black', label='model-fit')

ax.legend()

dydxs = []

dx = 0.1

for x in xms:
    num = func(x+dx, *params) - func(x, *params)
    denum = dx
    dydx = num/denum
    dydxs += [dydx]

ls = [[x,round(y,1)] for x,y in zip(xms[::3], dydxs[::3])]

In [ ]:
ls

In [ ]:
import pandas as pd
import os

In [ ]:
fdir = '../../Data/Calibration Data/Agrifood/Clean data'

fpath = os.path.join(fdir,'Yield/KBD - 2006.csv')
df = pd.read_csv(fpath)

df = df.set_index(pd.to_datetime(df['Month']))

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

s = df['ABW (kg)']

m = s.resample('M').max()

t0 = 2006
ts = np.array([x.year - t0  for x in m.index]) + np.array([(1/12)*x.month  for x in m.index])
ys = np.array(list(m.values))

f = lambda x, a, b, c: a*((x-c)/(1+((x-c)/b)))

g = lambda x, a, b, x0: a*(1 - np.exp(-b*(x-x0)))

params, _ = curve_fit(g, ts, ys, p0= [20, 0.14, 2])

print(t0, params)

tms = np.linspace(0,30,100)
ybfs = np.array([f(t,*params) for t in tms])

paramsm = [4, 6, 1.9]

yms = np.array([f(t,*paramsm) for t in tms])

ax.scatter(ts,ys)
#ax.plot(tms,ybfs, label='A')
ax.plot(tms, 1.8*g(tms, 33, 0.07, 1.2), label='A')
ax.plot(tms, 1.8*g(tms, 50, 0.04, 1.2), label='B')
ax.plot(tms, 1.8*g(tms, 21, 0.14, 2.2), label='C')

ax.set_xlim(0,30)

ax.legend()

In [ ]:
fig, (ax, ax2) = plt.subplots(2,1, figsize=(4,8))

s = df['BC (1/ha/mo)']

t0 = 2006

a, b, c = 25, 1.6, .3

def f(t, a, b, c):
    return (144/12)*a*(1+b*np.exp(-c*(t)))

def g(t, a, b, c):
    return (144/12)*a*(1+b*np.exp(-c*(t)))

m = s.resample('4M').max()

ts = [x.year - t0  for x in m.index]
ys = list(m.values)

params, _ = curve_fit(g, ts, ys)

ts.append(25)
ys.append(40)

tms = np.linspace(0,30,100)
yms = np.array([g(t,*params) for t in tms])

tms = np.linspace(0,30,100)
yfs = np.array([f(t,a,b,c) for t in tms])

ax.scatter(ts,ys)
ax.plot(tms,yfs, label='fronds')

# time between sex determination and bunch harvest
dt = 0

ax.plot(tms+dt,yms, label='fit to O')

yffs = yms/yfs

def sigmoid(t, k, t0, v1):
    return (1-v1)*1/(1 + np.exp(k*(t-t0))) + v1

k = 0.3
t0 = 12
v1 = 0.45

yfffs = np.array([sigmoid(t, k, t0, v1) for t in tms])

ax2.plot(tms, yffs, label='O')
ax2.plot(tms, 1.2*yffs, label='O+X%')
ax2.plot(tms, yfffs, label='M')

ax2.legend()
ax2.set_ylim(0, 1)

ax.plot(tms,yfs*yfffs, label='potential')

ax.legend()

In [ ]:
sig = lambda x,a,s,x0: a*(1/(1 + np.exp(-4*s*(x-x0)/a))) - a*(1/(1 + np.exp(-4*s*(0-x0)/a)))

sigf = lambda x,a,s,x0: s*(x-x0)+0.5*a - a*(1/(1 + np.exp(-4*s*(0-x0)/a)))

xs = np.linspace(0,1,100)

a = 0.4
x0 = .3
s = 1.5

ys = sig(xs, a, s, x0)

a2 = 0.3
x02 = .2
s2 = 1

y2s = sig(xs, a2, s2, x02)


plt.plot(xs,ys)
plt.plot(xs,y2s)
ax = plt.gca()
ax.set_xlim(0,1)
ax.set_ylim(0,1)

In [ ]:
f = lambda x,a,s : a*(1-np.exp(-s*x/a))

xs = np.linspace(0,300,100)
ys = f(xs,100,1)

plt.plot(xs,ys)

In [ ]:
sig = lambda x,a,s,x0: a*(1/(1 + np.exp(-4*s*(x-x0)/a))) - a*(1/(1 + np.exp(-4*s*(0-x0)/a)))

sigf = lambda x,a,s,x0: s*(x-x0)+0.5*a - a*(1/(1 + np.exp(-4*s*(0-x0)/a)))

xs = np.linspace(0,1,100)

a = 0.01
x0 = 0.2
s = 0.03

ys = sig(xs, a, s, x0)

a2 = 0.3
x02 = .2
s2 = 1

y2s = sig(xs, a2, s2, x02)


plt.plot(xs,30*ys)
#plt.plot(xs,y2s)
ax = plt.gca()